In [1]:
import os
import boto3
import pandas as pd

from io import BytesIO
from dotenv import load_dotenv

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

load_dotenv()

True

In [2]:
BUCKET = os.getenv("S3_BUCKET")
DATA_KEY = os.getenv("S3_OBJECT_KEY")
ENDPOINT_URL = os.getenv("S3_ENDPOINT_URL") or None

s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT_URL
)

response = s3.get_object(Bucket=BUCKET, Key=DATA_KEY)

In [3]:
df = pd.read_csv(BytesIO(response["Body"].read()))

print("Shape:", df.shape)
df.head()

Shape: (20151, 13)


,listing_id,snapshot_date,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type,price
0,2145993,2026-09-17,51.511100,-0.198250,2,1.0,1.0,1.0,4.0,Kensington and Chelsea,Entire rental unit,Entire home/apt,251.75
1,846381013872498937,2026-09-17,51.599540,-0.064500,6,1.5,3.0,4.0,2.0,Haringey,Entire home,Entire home/apt,201.60
2,1230990949123993577,2026-06-19,51.371535,-0.097551,2,1.0,NaN,1.0,1.0,Croydon,Private room in rental unit,Private room,73.00
3,54178450,2026-09-17,51.390740,-0.273480,2,1.0,1.0,1.0,1.0,Kingston upon Thames,Entire condo,Entire home/apt,126.00
4,1297347987655075325,2026-03-21,51.517398,0.012953,4,1.0,1.0,2.0,3.0,Newham,Entire rental unit,Entire home/apt,147.00


In [4]:
df = df.drop(columns=["listing_id", "snapshot_date"])

X = df.drop(columns=["price"])
y = df["price"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (16120, 10)
Test: (4031, 10)


In [6]:
num_cols = X_train.select_dtypes(include="number").columns
cat_cols = X_train.select_dtypes(exclude="number").columns

for col in num_cols:
    X_train[col] = X_train[col].fillna(X_train[col].median())
    X_test[col] = X_test[col].fillna(X_train[col].median())

for col in cat_cols:
    X_train[col] = X_train[col].fillna(X_train[col].mode()[0])
    X_test[col] = X_test[col].fillna(X_train[col].mode()[0])

In [7]:
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [8]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [9]:
y_pred = model.predict(X_test)

print(y_pred[:10])

[320.99820833 213.692      583.9267381  205.82368571  62.87766667
  97.11303333 147.1009     146.17523333  69.428925   242.29519286]


In [10]:
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
print("R2:", r2_score(y_test, y_pred))

MAE: 63.877022707260345
RMSE: 186.40118263480124
R2: 0.73337365814077
